In [3]:
import pandas as pd

DATA_PATH = "../../data/processed/matching_features_v2_1000.csv"
df = pd.read_csv(DATA_PATH)

df.shape

(50000, 22)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

feature_cols = [
    "job_skill_coverage",
    "skill_jaccard",
    "role_similarity",
    "text_similarity"
]

X = df[feature_cols]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

print("\nFEATURE IMPORTANCE")
print(
    pd.Series(
        model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)
)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      7884
           1       0.90      0.90      0.90      2116

    accuracy                           0.96     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.96      0.96      0.96     10000

ROC-AUC: 0.9925275785275915
PR-AUC: 0.9664834297172162

CONFUSION MATRIX
[[7665  219]
 [ 204 1912]]

FEATURE IMPORTANCE
role_similarity       0.462880
text_similarity       0.228429
job_skill_coverage    0.161008
skill_jaccard         0.147683
dtype: float64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

feature_sets = {
    "all": [
        "job_skill_coverage",
        "skill_jaccard",
        "role_similarity",
        "text_similarity"
    ],

    "role_only": [
        "role_similarity"
    ],

    "text_only": [
        "text_similarity"
    ],

    "skills_only": [
        "job_skill_coverage",
        "skill_jaccard"
    ]
}

results = []

for name, features in feature_sets.items():

    X = df[features]
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "feature_set": name,
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("ROC_AUC", ascending=False))

   feature_set        F1   ROC_AUC
0          all  0.900400  0.992528
1    role_only  0.668509  0.879594
3  skills_only  0.562595  0.776751
2    text_only  0.425492  0.715281


In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Deepu D\AI-hiring-system


In [7]:
from ml.v2.feature_generation import generate_relevance_score

In [8]:
df["relevance_score"] = df.apply(
    generate_relevance_score,
    axis=1
)

In [9]:
print(
    df["relevance_score"]
      .value_counts()
      .sort_index()
)

relevance_score
0     17350
1      8947
2      6515
3      6607
4      4519
5      3095
6      1742
7       737
8       321
9       129
10       36
11        2
Name: count, dtype: int64


In [10]:
print(
    df["relevance_score"]
      .value_counts(normalize=True)
      .sort_index()
      .mul(100)
      .round(2)
)

relevance_score
0     34.70
1     17.89
2     13.03
3     13.21
4      9.04
5      6.19
6      3.48
7      1.47
8      0.64
9      0.26
10     0.07
11     0.00
Name: proportion, dtype: float64


In [11]:
print(
    df.groupby("relevance_score")[
        [
            "job_skill_coverage",
            "skill_jaccard",
            "role_similarity",
            "text_similarity"
        ]
    ]
    .mean()
    .round(3)
)

                 job_skill_coverage  skill_jaccard  role_similarity  \
relevance_score                                                       
0                             0.000          0.000            0.003   
1                             0.045          0.023            0.029   
2                             0.050          0.024            0.147   
3                             0.072          0.036            0.215   
4                             0.102          0.051            0.320   
5                             0.137          0.067            0.415   
6                             0.180          0.091            0.467   
7                             0.244          0.128            0.522   
8                             0.320          0.175            0.602   
9                             0.361          0.201            0.665   
10                            0.479          0.286            0.551   
11                            0.500          0.286            0.625   

     

In [12]:
df["target_score"] = (
    df["relevance_score"] / 11.0
) * 100

### Train XGBoost regression

In [13]:
from xgboost import XGBRegressor

features = [
    "job_skill_coverage",
    "skill_jaccard",
    "role_similarity",
    "text_similarity"
]

X = df[features]
y = df["target_score"]

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets im

In [16]:
import numpy as np
pred = model.predict(X_test)

pred = np.clip(pred, 0, 100)

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 2.9169288476960222
RMSE: 3.6668962520450443
R²  : 0.9590389971479765


In [18]:
def get_match_level(score):

    if score < 20:
        return "Very Poor Match"

    elif score < 40:
        return "Poor Match"

    elif score < 60:
        return "Moderate Match"

    elif score < 75:
        return "Good Match"

    elif score < 90:
        return "Very Good Match"

    else:
        return "Excellent Match"

In [19]:
results = pd.DataFrame({
    "score": pred
})

results["match_level"] = results["score"].apply(
    get_match_level
)

## More analyzing

In [20]:
# ============================================================
# CURRENT FEATURES
# ============================================================

print("ALL COLUMNS:")
print("=" * 70)

for i, col in enumerate(df.columns):
    print(f"{i:2}. {col}")

ALL COLUMNS:
 0. job_id
 1. candidate_id
 2. job_title
 3. job_description
 4. job_skills
 5. job_role_family
 6. candidate_role
 7. candidate_skills
 8. candidate_role_family
 9. resume
10. skill_overlap_count
11. skill_overlap_ratio
12. candidate_skill_coverage
13. job_skill_coverage
14. skill_jaccard
15. role_similarity
16. candidate_experience
17. required_experience
18. experience_gap
19. text_similarity
20. label
21. relevance_score
22. target_score


In [21]:
print("\nNUMERIC COLUMNS:")
print("=" * 70)

print(df.select_dtypes(include=["number"]).columns.tolist())


NUMERIC COLUMNS:
['job_id', 'candidate_id', 'skill_overlap_count', 'skill_overlap_ratio', 'candidate_skill_coverage', 'job_skill_coverage', 'skill_jaccard', 'role_similarity', 'candidate_experience', 'required_experience', 'experience_gap', 'text_similarity', 'label', 'relevance_score', 'target_score']


In [22]:
# ============================================================
# EXPERIMENT 2 — ADDITIONAL CANDIDATE FEATURES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# ------------------------------------------------------------
# BASELINE FEATURES
# ------------------------------------------------------------

baseline_features = [
    "job_skill_coverage",
    "skill_jaccard",
    "role_similarity",
    "text_similarity",
]


# ------------------------------------------------------------
# ADDITIONAL FEATURES ALREADY AVAILABLE
# ------------------------------------------------------------

additional_features = [
    "skill_overlap_count",
    "skill_overlap_ratio",
    "candidate_skill_coverage",
    "candidate_experience",
    "experience_gap",
]


# ------------------------------------------------------------
# EXPANDED FEATURE SET
# ------------------------------------------------------------

expanded_features = baseline_features + additional_features


print("BASELINE FEATURES")
print("=" * 60)

for feature in baseline_features:
    print("-", feature)

print("\nEXPANDED FEATURES")
print("=" * 60)

for feature in expanded_features:
    print("-", feature)


# ------------------------------------------------------------
# PREPARE DATA
# ------------------------------------------------------------

X = df[expanded_features].copy()
y = df["target_score"].copy()

# Replace invalid numeric values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(0)


# ------------------------------------------------------------
# TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# TRAIN XGBOOST
# ------------------------------------------------------------

model_expanded = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_expanded.fit(X_train, y_train)


# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

pred_expanded = model_expanded.predict(X_test)

pred_expanded = np.clip(
    pred_expanded,
    0,
    100
)


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

mae = mean_absolute_error(y_test, pred_expanded)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_expanded
    )
)

r2 = r2_score(
    y_test,
    pred_expanded
)


print("\n")
print("=" * 60)
print("EXPANDED MODEL RESULTS")
print("=" * 60)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")


# ------------------------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------------------------

importance = pd.Series(
    model_expanded.feature_importances_,
    index=expanded_features
).sort_values(
    ascending=False
)

print("\nFEATURE IMPORTANCE")
print("=" * 60)

print(importance)

BASELINE FEATURES
- job_skill_coverage
- skill_jaccard
- role_similarity
- text_similarity

EXPANDED FEATURES
- job_skill_coverage
- skill_jaccard
- role_similarity
- text_similarity
- skill_overlap_count
- skill_overlap_ratio
- candidate_skill_coverage
- candidate_experience
- experience_gap


EXPANDED MODEL RESULTS
MAE : 2.8797
RMSE: 3.6404
R²  : 0.9596

FEATURE IMPORTANCE
role_similarity             0.414821
job_skill_coverage          0.215935
skill_overlap_ratio         0.206922
skill_overlap_count         0.063969
text_similarity             0.058401
skill_jaccard               0.032250
candidate_skill_coverage    0.003479
candidate_experience        0.003270
experience_gap              0.000954
dtype: float32


In [23]:
print("Total pairs:", len(df))
print("Unique candidates:", df["candidate_id"].nunique())

Total pairs: 50000
Unique candidates: 1000


In [24]:
# ============================================================
# REALISTIC SYNTHETIC GITHUB + LEETCODE FEATURES
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# ------------------------------------------------------------
# ONE PROFILE PER CANDIDATE
# ------------------------------------------------------------

candidate_ids = df["candidate_id"].unique()

candidate_features = pd.DataFrame({
    "candidate_id": candidate_ids
})

n = len(candidate_features)


# ============================================================
# LATENT CANDIDATE STRENGTH
# ============================================================
#
# This represents overall technical activity/strength.
#
# IMPORTANT:
# This is NOT given to the ML model.
# It is only used to generate realistic synthetic data.
# ============================================================

candidate_strength = np.random.beta(
    a=2.2,
    b=3.5,
    size=n
)


# ============================================================
# GITHUB
# ============================================================

# Public repositories
candidate_features["github_public_repos"] = np.maximum(
    0,
    np.round(
        2
        + candidate_strength * 35
        + np.random.normal(0, 4, n)
    )
).astype(int)


# Followers
candidate_features["github_followers"] = np.maximum(
    0,
    np.round(
        np.exp(
            2.0
            + candidate_strength * 4
            + np.random.normal(0, 1.0, n)
        )
    )
).astype(int)


# Stars
candidate_features["github_total_stars"] = np.maximum(
    0,
    np.round(
        np.exp(
            1.0
            + candidate_strength * 4
            + np.random.normal(0, 1.0, n)
        )
    )
).astype(int)


# Languages
candidate_features["github_language_diversity"] = np.clip(
    np.round(
        1
        + candidate_strength * 6
        + np.random.normal(0, 0.8, n)
    ),
    1,
    8
).astype(int)


# Active repositories
candidate_features["github_active_repos"] = np.maximum(
    0,
    np.round(
        candidate_features["github_public_repos"]
        * (
            0.25
            + candidate_strength * 0.65
            + np.random.normal(0, 0.08, n)
        )
    )
).astype(int)

candidate_features["github_active_repos"] = np.minimum(
    candidate_features["github_active_repos"],
    candidate_features["github_public_repos"]
)


# ============================================================
# LEETCODE
# ============================================================

# Total solved
candidate_features["leetcode_total_solved"] = np.clip(
    np.round(
        candidate_strength * 750
        + np.random.normal(0, 90, n)
    ),
    0,
    800
).astype(int)


total = candidate_features["leetcode_total_solved"]


# ------------------------------------------------------------
# Difficulty distribution
# ------------------------------------------------------------

hard_ratio = np.clip(
    0.03
    + candidate_strength * 0.12
    + np.random.normal(0, 0.015, n),
    0.01,
    0.20
)

medium_ratio = np.clip(
    0.30
    + candidate_strength * 0.20
    + np.random.normal(0, 0.04, n),
    0.20,
    0.65
)

candidate_features["leetcode_hard"] = (
    total * hard_ratio
).astype(int)

candidate_features["leetcode_medium"] = (
    total * medium_ratio
).astype(int)

candidate_features["leetcode_easy"] = np.maximum(
    0,
    total
    - candidate_features["leetcode_hard"]
    - candidate_features["leetcode_medium"]
)


# ============================================================
# CONTEST
# ============================================================

candidate_features["leetcode_contest_rating"] = np.clip(
    np.round(
        1200
        + candidate_strength * 850
        + np.random.normal(0, 120, n)
    ),
    1000,
    2400
).astype(int)


# Better candidates tend to have better percentile
candidate_features["leetcode_contest_percentile"] = np.clip(
    100
    - candidate_strength * 90
    + np.random.normal(0, 8, n),
    1,
    100
)

candidate_features["leetcode_contest_percentile"] = np.round(
    candidate_features["leetcode_contest_percentile"],
    2
)


candidate_features["leetcode_contests_attended"] = np.maximum(
    0,
    np.round(
        candidate_strength * 25
        + np.random.normal(0, 4, n)
    )
).astype(int)


# ============================================================
# CONSISTENCY
# ============================================================

candidate_features["leetcode_streak"] = np.clip(
    np.round(
        candidate_strength * 180
        + np.random.normal(0, 30, n)
    ),
    0,
    365
).astype(int)


candidate_features["leetcode_active_days"] = np.clip(
    np.round(
        candidate_strength * 300
        + np.random.normal(0, 40, n)
    ),
    0,
    365
).astype(int)


# ============================================================
# DSA CATEGORY STRENGTH
# ============================================================

candidate_features["dp_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + candidate_strength * 0.12
            + np.random.normal(0, 0.015, n)
        )
    )
).astype(int)


candidate_features["graph_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + candidate_strength * 0.10
            + np.random.normal(0, 0.015, n)
        )
    )
).astype(int)


candidate_features["greedy_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.02
            + candidate_strength * 0.08
            + np.random.normal(0, 0.012, n)
        )
    )
).astype(int)


candidate_features["tree_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + candidate_strength * 0.10
            + np.random.normal(0, 0.015, n)
        )
    )
).astype(int)


candidate_features["binary_search_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.02
            + candidate_strength * 0.07
            + np.random.normal(0, 0.01, n)
        )
    )
).astype(int)


# ============================================================
# REMOVE LATENT VARIABLE
# ============================================================

# candidate_strength is ONLY for generating synthetic data.
# It must NEVER become an ML feature.

# It isn't stored in candidate_features, so we're safe.


# ============================================================
# MERGE WITH PAIR DATASET
# ============================================================

df_ml = df.merge(
    candidate_features,
    on="candidate_id",
    how="left"
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("SYNTHETIC CANDIDATE DATA")
print("=" * 70)

print(f"Original rows : {len(df):,}")
print(f"New rows      : {len(df_ml):,}")
print(f"Candidates    : {candidate_features['candidate_id'].nunique():,}")

assert len(df_ml) == len(df)

print("\nFEATURES:")
print(
    candidate_features
    .drop(columns=["candidate_id"])
    .describe()
    .round(2)
)

SYNTHETIC CANDIDATE DATA
Original rows : 50,000
New rows      : 50,000
Candidates    : 1,000

FEATURES:
       github_public_repos  github_followers  github_total_stars  \
count              1000.00           1000.00             1000.00   
mean                 15.55             75.77               28.77   
std                   7.61            174.76               56.24   
min                   0.00              1.00                0.00   
25%                  10.00             16.00                6.00   
50%                  15.00             35.00               12.00   
75%                  21.00             79.00               30.00   
max                  38.00           4080.00              890.00   

       github_language_diversity  github_active_repos  leetcode_total_solved  \
count                    1000.00              1000.00                1000.00   
mean                        3.36                 8.68                 297.74   
std                         1.41           

In [25]:
print("\nCORRELATION WITH TOTAL SOLVED")
print("=" * 70)

print(
    candidate_features[
        [
            "github_public_repos",
            "github_followers",
            "github_total_stars",
            "github_language_diversity",
            "github_active_repos",
            "leetcode_total_solved",
            "leetcode_contest_rating",
            "leetcode_streak",
            "leetcode_active_days",
            "dp_strength",
            "graph_strength",
            "greedy_strength",
        ]
    ].corr()["leetcode_total_solved"]
    .sort_values(ascending=False)
)


CORRELATION WITH TOTAL SOLVED
leetcode_total_solved        1.000000
dp_strength                  0.934574
graph_strength               0.923402
greedy_strength              0.914939
github_active_repos          0.766506
github_public_repos          0.742547
leetcode_active_days         0.695639
leetcode_contest_rating      0.672913
github_language_diversity    0.661631
leetcode_streak              0.619017
github_total_stars           0.346357
github_followers             0.266867
Name: leetcode_total_solved, dtype: float64


Final update for github and leetcode

In [26]:
# ============================================================
# JOB-AWARE REALISTIC SYNTHETIC GITHUB + LEETCODE FEATURES
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# ------------------------------------------------------------
# ONE PROFILE PER CANDIDATE
# ------------------------------------------------------------

candidate_ids = df["candidate_id"].unique()

candidate_features = pd.DataFrame({
    "candidate_id": candidate_ids
})

n_candidates = len(candidate_features)


# ============================================================
# STEP 1 — BASE CANDIDATE STRENGTH
# ============================================================

candidate_strength = np.random.beta(
    a=2.2,
    b=3.5,
    size=n_candidates
)

candidate_strength_df = pd.DataFrame({
    "candidate_id": candidate_ids,
    "_candidate_strength": candidate_strength
})


# ============================================================
# STEP 2 — ESTIMATE MATCH STRENGTH PER CANDIDATE
# ============================================================
#
# We use the EXISTING matching features only to make the
# synthetic external signals somewhat job-aware.
#
# This variable is NOT given to the ML model.
# ============================================================

match_strength = (
    0.30 * df["job_skill_coverage"].fillna(0)
    + 0.20 * df["skill_jaccard"].fillna(0)
    + 0.35 * df["role_similarity"].fillna(0)
    + 0.15 * df["text_similarity"].fillna(0)
)

candidate_match = (
    df.assign(_match_strength=match_strength)
      .groupby("candidate_id")["_match_strength"]
      .mean()
      .reindex(candidate_ids)
      .fillna(0)
      .values
)


# Normalize approximately to 0–1
candidate_match = np.clip(
    candidate_match / 0.35,
    0,
    1
)


# ------------------------------------------------------------
# COMBINE GENERAL STRENGTH + JOB MATCH
# ------------------------------------------------------------

technical_strength = (
    0.55 * candidate_strength
    + 0.45 * candidate_match
    + np.random.normal(0, 0.08, n_candidates)
)

technical_strength = np.clip(
    technical_strength,
    0,
    1
)


# ============================================================
# GITHUB
# ============================================================

candidate_features["github_public_repos"] = np.clip(
    np.round(
        2
        + technical_strength * 35
        + np.random.normal(0, 4, n_candidates)
    ),
    0,
    None
).astype(int)


candidate_features["github_followers"] = np.maximum(
    0,
    np.round(
        np.exp(
            2
            + technical_strength * 4
            + np.random.normal(0, 1.0, n_candidates)
        )
    )
).astype(int)


candidate_features["github_total_stars"] = np.maximum(
    0,
    np.round(
        np.exp(
            1
            + technical_strength * 4
            + np.random.normal(0, 1.0, n_candidates)
        )
    )
).astype(int)


candidate_features["github_language_diversity"] = np.clip(
    np.round(
        1
        + technical_strength * 6
        + np.random.normal(0, 0.8, n_candidates)
    ),
    1,
    8
).astype(int)


candidate_features["github_active_repos"] = np.maximum(
    0,
    np.round(
        candidate_features["github_public_repos"]
        * (
            0.25
            + technical_strength * 0.65
            + np.random.normal(0, 0.08, n_candidates)
        )
    )
).astype(int)

candidate_features["github_active_repos"] = np.minimum(
    candidate_features["github_active_repos"],
    candidate_features["github_public_repos"]
)


# ============================================================
# LEETCODE
# ============================================================

candidate_features["leetcode_total_solved"] = np.clip(
    np.round(
        technical_strength * 750
        + np.random.normal(0, 90, n_candidates)
    ),
    0,
    800
).astype(int)


total = candidate_features["leetcode_total_solved"]


# ------------------------------------------------------------
# Difficulty
# ------------------------------------------------------------

hard_ratio = np.clip(
    0.03
    + technical_strength * 0.12
    + np.random.normal(0, 0.015, n_candidates),
    0.01,
    0.20
)

medium_ratio = np.clip(
    0.30
    + technical_strength * 0.20
    + np.random.normal(0, 0.04, n_candidates),
    0.20,
    0.65
)

candidate_features["leetcode_hard"] = (
    total * hard_ratio
).astype(int)

candidate_features["leetcode_medium"] = (
    total * medium_ratio
).astype(int)

candidate_features["leetcode_easy"] = np.maximum(
    0,
    total
    - candidate_features["leetcode_hard"]
    - candidate_features["leetcode_medium"]
)


# ============================================================
# CONTEST
# ============================================================

candidate_features["leetcode_contest_rating"] = np.clip(
    np.round(
        1200
        + technical_strength * 850
        + np.random.normal(0, 120, n_candidates)
    ),
    1000,
    2400
).astype(int)


candidate_features["leetcode_contest_percentile"] = np.clip(
    100
    - technical_strength * 90
    + np.random.normal(0, 8, n_candidates),
    1,
    100
)

candidate_features["leetcode_contest_percentile"] = np.round(
    candidate_features["leetcode_contest_percentile"],
    2
)


candidate_features["leetcode_contests_attended"] = np.maximum(
    0,
    np.round(
        technical_strength * 25
        + np.random.normal(0, 4, n_candidates)
    )
).astype(int)


# ============================================================
# CONSISTENCY
# ============================================================

candidate_features["leetcode_streak"] = np.clip(
    np.round(
        technical_strength * 180
        + np.random.normal(0, 30, n_candidates)
    ),
    0,
    365
).astype(int)


candidate_features["leetcode_active_days"] = np.clip(
    np.round(
        technical_strength * 300
        + np.random.normal(0, 40, n_candidates)
    ),
    0,
    365
).astype(int)


# ============================================================
# DSA SIGNALS
# ============================================================

candidate_features["dp_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + technical_strength * 0.12
            + np.random.normal(0, 0.015, n_candidates)
        )
    )
).astype(int)


candidate_features["graph_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + technical_strength * 0.10
            + np.random.normal(0, 0.015, n_candidates)
        )
    )
).astype(int)


candidate_features["greedy_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.02
            + technical_strength * 0.08
            + np.random.normal(0, 0.012, n_candidates)
        )
    )
).astype(int)


candidate_features["tree_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.03
            + technical_strength * 0.10
            + np.random.normal(0, 0.015, n_candidates)
        )
    )
).astype(int)


candidate_features["binary_search_strength"] = np.maximum(
    0,
    np.round(
        total * (
            0.02
            + technical_strength * 0.07
            + np.random.normal(0, 0.01, n_candidates)
        )
    )
).astype(int)


# ============================================================
# MERGE
# ============================================================

df_ml = df.merge(
    candidate_features,
    on="candidate_id",
    how="left"
)


# ============================================================
# VALIDATION
# ============================================================

assert len(df_ml) == len(df)

print("=" * 70)
print("JOB-AWARE SYNTHETIC DATA")
print("=" * 70)

print(f"Pairs              : {len(df_ml):,}")
print(f"Unique candidates  : {df_ml['candidate_id'].nunique():,}")

print("\nCandidate features:")
print(candidate_features.head())

print("\nFeature summary:")
print(
    candidate_features
    .drop(columns=["candidate_id"])
    .describe()
    .round(2)
)

JOB-AWARE SYNTHETIC DATA
Pairs              : 50,000
Unique candidates  : 1,000

Candidate features:
   candidate_id  github_public_repos  github_followers  github_total_stars  \
0          4216                   10                32                   5   
1          4289                   22                32                   8   
2          3859                   20                55                   3   
3          2654                   15                28                  37   
4          5390                   15                31                   4   

   github_language_diversity  github_active_repos  leetcode_total_solved  \
0                          1                    4                    110   
1                          3                   14                    320   
2                          5                    9                    406   
3                          3                    6                    135   
4                          3                    7 

#### Now training with github and leetcode signals

In [27]:
# ============================================================
# GITHUB + LEETCODE MODEL
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

base_features = [
    "job_skill_coverage",
    "skill_jaccard",
    "role_similarity",
    "text_similarity",
    "skill_overlap_count",
    "skill_overlap_ratio",
    "candidate_skill_coverage",
    "candidate_experience",
    "experience_gap",
]


github_features = [
    "github_public_repos",
    "github_followers",
    "github_total_stars",
    "github_language_diversity",
    "github_active_repos",
]


leetcode_features = [
    "leetcode_total_solved",
    "leetcode_easy",
    "leetcode_medium",
    "leetcode_hard",
    "leetcode_contest_rating",
    "leetcode_contest_percentile",
    "leetcode_contests_attended",
    "leetcode_streak",
    "leetcode_active_days",
    "dp_strength",
    "graph_strength",
    "greedy_strength",
    "tree_strength",
    "binary_search_strength",
]


all_features = (
    base_features
    + github_features
    + leetcode_features
)


# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

X = df_ml[all_features].copy()
y = df_ml["target_score"].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)


# ------------------------------------------------------------
# SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

model_github_lc = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_github_lc.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

pred = model_github_lc.predict(X_test)

pred = np.clip(
    pred,
    0,
    100
)


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)

r2 = r2_score(
    y_test,
    pred
)


print("=" * 70)
print("GITHUB + LEETCODE MODEL")
print("=" * 70)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")


# ------------------------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------------------------

importance = pd.Series(
    model_github_lc.feature_importances_,
    index=all_features
).sort_values(
    ascending=False
)

print("\nFEATURE IMPORTANCE")
print("=" * 70)

print(importance)

GITHUB + LEETCODE MODEL
MAE : 2.5802
RMSE: 3.3666
R²  : 0.9655

FEATURE IMPORTANCE
skill_overlap_ratio            0.440197
role_similarity                0.247802
skill_overlap_count            0.104660
job_skill_coverage             0.103501
skill_jaccard                  0.041621
text_similarity                0.034474
candidate_skill_coverage       0.004591
experience_gap                 0.001467
candidate_experience           0.001446
leetcode_contest_percentile    0.001226
leetcode_hard                  0.001218
graph_strength                 0.001194
leetcode_active_days           0.001189
leetcode_streak                0.001174
leetcode_contest_rating        0.001147
leetcode_medium                0.001141
tree_strength                  0.001126
binary_search_strength         0.001084
greedy_strength                0.001054
leetcode_total_solved          0.001045
leetcode_contests_attended     0.001023
dp_strength                    0.001020
github_total_stars             0.0009

In [28]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models/job_matcher.joblib")

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    model_github_lc,
    MODEL_PATH
)

print(f"Model saved to: {MODEL_PATH.resolve()}")

Model saved to: C:\Deepu D\AI-hiring-system\ml\v2\models\job_matcher.joblib


In [37]:
X

,job_skill_coverage,skill_jaccard,role_similarity,text_similarity,skill_overlap_count,skill_overlap_ratio,candidate_skill_coverage,candidate_experience,experience_gap,github_public_repos,...,leetcode_contest_rating,leetcode_contest_percentile,leetcode_contests_attended,leetcode_streak,leetcode_active_days,dp_strength,graph_strength,greedy_strength,tree_strength,binary_search_strength
0,0.00,0.000000,0.166667,0.047528,0.0,0.00,0.000000,5.0,5.0,10,...,1426,77.68,7,59,109,6,5,5,8,4
1,0.00,0.000000,0.400000,0.087454,0.0,0.00,0.000000,5.0,5.0,22,...,1657,67.00,14,5,80,23,21,18,32,17
2,0.00,0.000000,0.400000,0.070310,0.0,0.00,0.000000,5.0,5.0,20,...,1696,68.35,5,117,138,28,30,28,30,26
3,0.00,0.000000,0.400000,0.055727,0.0,0.00,0.000000,0.0,0.0,15,...,1440,82.76,7,0,38,9,5,6,7,3
4,0.00,0.000000,0.166667,0.024276,0.0,0.00,0.000000,0.0,0.0,15,...,1695,56.35,15,80,141,8,7,5,5,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0.00,0.000000,0.333333,0.071321,0.0,0.00,0.000000,5.0,5.0,6,...,1339,80.15,8,33,6,1,2,1,1,1
49996,0.25,0.055556,0.000000,0.077853,1.0,0.25,0.066667,0.0,0.0,17,...,1776,60.94,10,79,107,25,23,5,19,11
49997,0.00,0.000000,0.333333,0.027282,0.0,0.00,0.000000,0.0,0.0,12,...,1524,66.31,12,29,108,16,18,7,10,15
49998,0.00,0.000000,0.000000,0.119982,0.0,0.00,0.000000,5.0,5.0,16,...,1229,80.27,6,46,69,14,9,9,11,10
